## NLP sentiment analysis

Sentiment analysis is the use of computers to interpret the underlying subjective tone of a piece of text and categorize it into positive, neutral, or negative classes.

For this project, sentiment analysis is useful because customer reviews often contain a lot of information about product quality, texture, skin compatibility, packaging, and overall satisfaction. I want to understand whether people feel positively or negatively about a product, and whether the sentiment is linked to specific product attributes or repeated complaints.

My goal is to turn raw review text into useful business insight. Instead of looking at each review individually, I will try to summarize the overall tone of customer feedback, identify which products generate the strongest reactions, and highlight recurring reasons behind positive or negative sentiment. This will help me understand customer experience more clearly and support better product recommendations and analysis.


Using selected columns from reviews dataframe, I can start with sentiment analysis.

### Additional Insights to Explore

    ### Product & Brand Level
    - **Top/bottom rated products and brands** — aggregate sentiment scores by `product_name`/`brand` to find consistent winners and losers
    - **Sentiment vs. star rating mismatch** — reviews where text sentiment disagrees with the numeric rating (e.g., positive text but 1-star rating) — often reveals sarcasm, mislabeled ratings, or shipping/packaging complaints
    unrelated to the product itself
    - **Category-level trends** — compare sentiment across categories (skincare vs. makeup vs. haircare) to see which categories have happier customers

    ### Aspect-Based Sentiment Analysis
    - **Extract specific product attributes** mentioned in reviews (texture, scent, packaging, price, longevity, shade range) and compute sentiment per aspect
    - **"Value for money" sentiment** — isolate mentions of price/cost and see if sentiment skews negative even when overall sentiment is positive

    ### Customer Behavior / Demographics (if available)
    - **Skin type / skin tone correlation** — do reviews from certain skin types or tones trend more negative for specific products (useful for shade-matching or formulation issues)?
    - **Verified purchase vs. non-verified sentiment** — do verified purchasers rate more critically/honestly?
    - **Review recency trends** — has sentiment for a product changed over time (e.g., after a reformulation or price change)?


## Imports and data

In [7]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [8]:
reviews = pd.read_csv('reviews.csv', )
reviews.head()

,author_id,rating,helpfulness,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,1741593524,5,1.0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,Doesn't apply,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,31423088263,1,NaN,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,Doesn't apply,Doesn't apply,Doesn't apply,Doesn't apply,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,5061282401,5,NaN,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,6083038851,5,NaN,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,Doesn't apply,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,47056667835,5,NaN,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,Doesn't apply,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [9]:
reviews.shape

(1503850, 15)

In [10]:
data = reviews[['author_id','rating', 'review_text', 'review_title', 'product_name']]
data.head()

,author_id,rating,review_text,review_title,product_name
0,1741593524,5,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,Gentle Hydra-Gel Face Cleanser
1,31423088263,1,I bought this lip mask after reading the revie...,Disappointed,Lip Sleeping Mask Intense Hydration with Vitam...
2,5061282401,5,My review title says it all! I get so excited ...,New Favorite Routine,Lip Sleeping Mask Intense Hydration with Vitam...
3,6083038851,5,I’ve always loved this formula for a long time...,Can't go wrong with any of them,Lip Sleeping Mask Intense Hydration with Vitam...
4,47056667835,5,"If you have dry cracked lips, this is a must h...",A must have !!!,Lip Sleeping Mask Intense Hydration with Vitam...


In [11]:
%pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 35.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 18.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: /Users/gabrielaslomiany/PyCharmMiscProject/Sephora/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import nltk
# nltk.download('punkt_tab')
#
# text = "NLP is fun. Let's dive into it!"
# sentences =nltk.sent_tokenize(text)
#
# # The result is a list of sentences that we can analyze individually.
# print(sentences)